初始化

In [ ]:
import sqlite3
import os
import logging

DB_DIR = "C:\github\East-Coast-Gas-Daily-Snapshot/data/processed"
DB_PATH = os.path.join(DB_DIR, "east_coast_gas.db")

def init_db():
    """初始化升级版 SQLite 数据库（支持审计追踪与严格分层）"""
    os.makedirs(DB_DIR, exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # 1. Prices: 增加审计列
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            gas_date TEXT,
            hub TEXT,
            price_type TEXT, 
            price REAL,
            source_file TEXT,
            ingested_at TEXT,
            PRIMARY KEY (gas_date, hub, price_type)
        )
    ''')

    # 2. Storage: 主键彻底改为 facility_id
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS storage (
            gas_date TEXT,
            facility_id INTEGER,
            held_in_storage REAL,
            source_file TEXT,
            ingested_at TEXT,
            PRIMARY KEY (gas_date, facility_id)
        )
    ''')

    # 3. Flows: 增加 facility_type 鉴别列，主键改为 facility_id
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS flows (
            gas_date TEXT,
            facility_id INTEGER,
            facility_type TEXT,
            demand REAL,       
            supply REAL,
            transfer_in REAL,
            transfer_out REAL,
            source_file TEXT,
            ingested_at TEXT,
            PRIMARY KEY (gas_date, facility_id)
        )
    ''')

    conn.commit()
    conn.close()
    logging.info(f"Database schema initialized with Data Lineage fields at {DB_PATH}")

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    init_db()

升级审计表

In [ ]:
import sqlite3
import os

DB_PATH = os.path.join("data", "processed", "east_coast_gas.db")

def init_audit_system():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print("🛠️  正在创建数据修订审计表 (data_revisions)...")
    # 1. 创建审计表
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS data_revisions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name TEXT NOT NULL,         -- 'storage' 或 'flows'
            gas_date TEXT NOT NULL,           -- 被修改的数据业务日期 (Event Time)
            facility_id INTEGER NOT NULL,     -- 关联的设施ID
            metric_name TEXT NOT NULL,        -- 被修改的指标名 (如 'held_in_storage', 'supply' 等)
            old_value REAL,                   -- 修改前的旧数值
            new_value REAL,                   -- 修改后的新数值
            changed_at TEXT NOT NULL          -- 审计记录发生的时间 (Processing Time)
        )
    ''')

    print("⚙️  正在为 storage 表挂载自动化修订触发器...")
    # 2. 为 storage 表创建触发器：当 held_in_storage 发生变化时自动触发
    cursor.execute('''
        CREATE TRIGGER IF NOT EXISTS audit_storage_update
        AFTER UPDATE ON storage
        FOR EACH ROW
        WHEN OLD.held_in_storage <> NEW.held_in_storage
        BEGIN
            INSERT INTO data_revisions (table_name, gas_date, facility_id, metric_name, old_value, new_value, changed_at)
            VALUES ('storage', OLD.gas_date, OLD.facility_id, 'held_in_storage', OLD.held_in_storage, NEW.held_in_storage, datetime('now'));
        END;
    ''')

    print("⚙️  正在为 flows 表挂载自动化修订触发器...")
    # 3. 为 flows 表创建触发器：分别监控 supply, transfer_in, transfer_out 变动
    cursor.execute('''
        CREATE TRIGGER IF NOT EXISTS audit_flows_supply_update
        AFTER UPDATE ON flows
        FOR EACH ROW
        WHEN OLD.supply <> NEW.supply
        BEGIN
            INSERT INTO data_revisions (table_name, gas_date, facility_id, metric_name, old_value, new_value, changed_at)
            VALUES ('flows', OLD.gas_date, OLD.facility_id, 'supply', OLD.supply, NEW.supply, datetime('now'));
        END;
    ''')

    cursor.execute('''
        CREATE TRIGGER IF NOT EXISTS audit_flows_transfer_in_update
        AFTER UPDATE ON flows
        FOR EACH ROW
        WHEN OLD.transfer_in <> NEW.transfer_in
        BEGIN
            INSERT INTO data_revisions (table_name, gas_date, facility_id, metric_name, old_value, new_value, changed_at)
            VALUES ('flows', OLD.gas_date, OLD.facility_id, 'transfer_in', OLD.transfer_in, NEW.transfer_in, datetime('now'));
        END;
    ''')

    cursor.execute('''
        CREATE TRIGGER IF NOT EXISTS audit_flows_transfer_out_update
        AFTER UPDATE ON flows
        FOR EACH ROW
        WHEN OLD.transfer_out <> NEW.transfer_out
        BEGIN
            INSERT INTO data_revisions (table_name, gas_date, facility_id, metric_name, old_value, new_value, changed_at)
            VALUES ('flows', OLD.gas_date, OLD.facility_id, 'transfer_out', OLD.transfer_out, NEW.transfer_out, datetime('now'));
        END;
    ''')
    cursor.execute('''
        CREATE TRIGGER IF NOT EXISTS audit_flows_demand_update
        AFTER UPDATE ON flows
        FOR EACH ROW
        WHEN OLD.demand <> NEW.demand
        BEGIN
            INSERT INTO data_revisions (table_name, gas_date, facility_id, metric_name, old_value, new_value, changed_at)
            VALUES ('flows', OLD.gas_date, OLD.facility_id, 'demand', OLD.demand, NEW.demand, datetime('now'));
        END;
    ''')
    conn.commit()
    conn.close()
    print("✅ 数据库审计追踪系统部署完毕！触发器已在底层隐形实时监控。")

if __name__ == "__main__":
    init_audit_system()

flows 增加demand列

In [ ]:
import sqlite3
import os

DB_PATH = os.path.join("data", "processed", "east_coast_gas.db")

def add_column():
    print("⚙️ 正在连接数据库...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    try:
        # 使用 ALTER TABLE 为 flows 表新增 demand 列
        # DEFAULT 0.0 保证了历史旧数据不会变成讨厌的 NULL，而是默认填充 0
        cursor.execute("ALTER TABLE flows ADD COLUMN demand REAL DEFAULT 0.0;")
        print("✅ 大成功！已经在 flows 表中成功添加 'demand' 列。")
        
    except sqlite3.OperationalError as e:
        # 捕捉列已存在的报错，防止脚本炸毁
        if "duplicate column name" in str(e).lower():
            print("⚠️ 'demand' 列已经存在了，无需重复添加。")
        else:
            print(f"❌ 发生未知错误: {e}")
            
    conn.commit()
    conn.close()

if __name__ == "__main__":
    add_column()

In [2]:


import sqlite3
import os

DB_PATH = os.path.join("data", "processed", "east_coast_gas.db")

def add_table():
    print("⚙️ 正在连接数据库...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("CREATE TABLE dwgm_prices (gas_date TEXT NOT NULL,price_6am_schedule REAL,approval_datetime TEXT,is_forecast INTEGER,source_file TEXT,ingested_at TEXT,UNIQUE(gas_date));")
    print("✅ 大成功！")
            
    conn.commit()
    conn.close()

if __name__ == "__main__":
    add_table()


⚙️ 正在连接数据库...
✅ 大成功！


In [3]:




import sqlite3
import os

DB_PATH = os.path.join("data", "processed", "east_coast_gas.db")

def add_table():
    print("⚙️ 正在连接数据库...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
        CREATE TABLE weather (
        date TEXT NOT NULL, city TEXT NOT NULL,
        temp_max REAL, temp_min REAL, temp_mean REAL, hdd REAL,
        is_forecast INTEGER, source TEXT, ingested_at TEXT,
        UNIQUE(date, city));
    ''')
    print("✅ 大成功！")
            
    conn.commit()
    conn.close()

if __name__ == "__main__":
    add_table()

⚙️ 正在连接数据库...
✅ 大成功！
